RNN PER TESTO: TRASFORMARE SIMBOLI IN SEQUENZE INTELLIGENTI

Punto di passaggio tra Deep Learning classico e il Natural Language Processing (NLP).

Una CNN vede un'immagine come una griglia di pixel.
Una frase, invece, è una sequenza
    il -> film -> è -> bello
L'ordine è fondamentale.
Cambiando l'ordine il siginificato potrebbe cambiare di molto
Quindi la rete deve ricordare l'ordine.
Nascono così le RNN (Recurrent Neural Network)

In una RNN il testo è letto una parola alla volta.
non legge tutta la frase insieme.
Legge una parola - > la elabora -> la ricorda -> passa alla successiva

Ma il computer non capisce le parole
Per una rete 'gatto' non significa nulla, bisogna trasformalo in numeri

* Step 1 - Tokenizzazione
["il","film","è","bello"]
questi token possono essere parole intere, sotto-parole o singoli caratteri

* Step 2 - Vocabolario
Ora costruiamo un dizionario
il              numero 1
film            numero 2
è               numero 3
bello           numero 4
e               numero 5
è               numero 3
interessante    numero 6
la frese diventa [1,2,3,4,5,3,6]
Ora le rete può lavorare
Film vale 2, ma è un semplice identificatore
Attenzione, il codice (numero 1, numero 2, ecc) non contiene informazioni, è solo un codice che serve solo ad identifcare un token

Rimozione del rumore: in molti dataset è necessario rimuovere tag, html, caratteri speciali o spazi multipli che non aggiungono valore
Le sequenze di numeri devono essere uniformate tramite l'aggiunta di token neutri
Quindi il vocabolario è l'insieme finito di tutti i token unici presenti nel corpus di addestramento. Creare un vocabolario significa stabilire una corrispondenza biunivoca tra un simbolo testuale e un valore numerico unico
- Char - to - int: creazione di un dizionario python per convertire ogni caratteri in un numero
- Int - to -Char: creazione di un dizionario inverso per tradurre le predizioni numeriche del modello in testo intelligente.
- Token OOV: gestione dei token 'out of vocabulary'. esempio se utilizzo solo alfabeto latino, quando ricevo caretteri cinesi devo poter dire 'non lo conosco'

Ricordardi di salvere sempre il vocabolario per la traduzione del risultato

* Step 3 - Embedding
non possiamo dare alla rete il numero 2 (film) ma dobbiamo trasformarlo in un vettore
Se dovessimo dare alla rete un numero (2=film) potrebbe capire che 2>1 e 2<3 quindi capire che film è minore di bello. oppure potrebbe pensare che 2-1=1 quindi film - il = il
Allora usiamo un vettore di numeri  esempio [0.15,-0.32,0.88,0.01]

Questo vettore rappresenta il significato della parola. 
O meglio: il vettore rappresenta come la rete ha imparato ad usare quella parola osservando milioni di frasi

Chi ci decide quei numeri?
non li scegliamo noi, li impara la rete

E come li scegliere la rete?
gatto e cane avranno dei numeri molto simili (la rete ha imparato che gatto e cane sono simili)
Ma come fa la rete a capire che gatto e cane sono simili
Osserva la rete globale e vede che gatto e cane appaiono spesso in contesti simili. Quindi sposta lentamente i loro vettori finchè diventano vicini
E' come il cervello umano, gatto è associato a diverse parole (animale, mammifero, ha la coda, ha 4 zampe, ecc). Il cervello umano non usa una parole, usa un indieme di caratteristiche
Il computer fa una cosa simile

Perchè non basta un solo numero ma sono necessari vettori di numeri?
perchè la parola gatto potrebbe essere descritto da caratteristiche come: animale, domestrico, carnivoro, mammifero, ecc
Ovviamente la rete non sa che una colonna (del vettore) rappresenta 'animale' l'altra parola rappresenta 'domestico' ecc. Sono concettoi che scopre da sola

Il processo di trasformazione di una parola in vettore di numeri si chiama 'Embedding' ed è uno strato della rete neurale
in Keras è scritto:
Embedding(input_dim=10000,output_dim=128)
Con l'embedding pertanto è la rete, che da sola, impara una rappresentazione numerica del significato delle parole
Le parole che appaiono in contesti simili avranno vettori simili, questo è il principio su cui si basa Word2Vec
Ma n gatto=2 non sappiamo 2 cosa rappresenta, potrebbe rappresentare un animale, le orecchie, i baffi, ecc
Ogni numero, preso da solo, non ha un significato interpretabile, è l'insieme dei numeri (128 o 256 o 768) che rappresenta il significato della parola gatto.
Quindi la rete non riceve il significato delle parole, lo scopre osservando come vengono usate.

* Step 4 - Hidden State
l'hidden state è la memoria della rete
Immaginalo come un blocco note.
dopo aver letto la prima parola (in vettore di numeri) il blocco note contiene la prima parola
poi legge la seconda parola (in vettore di numeri) il blocco note contiene la prima e la seconda parola.
ecc
Il blocco note cresce continuamente
Utilizza sempre la stessa rete (Recurrent) non costruisce una rete nuova per ogni parola
parola1 -> rete -> memoria -> parola2 -> stessa rete -> nuova memoria -> parola3 -> stessa rete -> nuova memoria

Problema delle RNN
dopo molte parole della frase la rete dovrebbe ricordarsi una parola all'inizio della frease, ma ne frattempo ha elaborato centinaia di parole e la memoria degrada, questo fenomeno di chiama Vanishing Gradient

Ecco che arrivano le LSTM e GRU
RNN: parola -> memoria -> parola -> memoria -> parola -> memoria
LSTM: Gate -> memoria lunga -> decisioni
Le GRU fanno la stessa cosa delle LSTM ma in modo più semplice.

Oggi è raro costruire nuovi sistemi NLP basati su RNN. Le RNN hanno introdotto il concetto fondamentale di elaborazione sequenziale, ma presentano limiti nel gestire dipendenze molto lunghe. Le LSTM e le GRU hanno migliorato questi aspetti, e successivamente i TRANSFORMER (su cui si basano modelli come BERT e GPT) hanno rivoluzionato il campo permettendo di elaborare tutte le parole della sequenza in parallelo e di catturare relazioni a lunga distanza in modo molto più semplice.



In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# =================================================================
# 1. PREPARAZIONE DEL DATASET (Tokenizzazione a livello di carattere)
# =================================================================
# Creiamo un dataset fittizio ripetendo una stringa.
text_data = "l'intelligenza artificiale impara dai dati e genera nuova conoscenza. " * 10

# Estraiamo i caratteri unici (il nostro vocabolario)
vocab = sorted(list(set(text_data)))
# char_to_idx: mappa ogni carattere a un numero (necessario perché le reti neurali leggono solo numeri)
char_to_idx = {char: i for i, char in enumerate(vocab)}
# idx_to_char: mappa i numeri ai caratteri (necessario per tradurre l'output della rete in testo leggibile)
idx_to_char = {i: char for i, char in enumerate(vocab)}

vocab_size = len(vocab)
print(f"Dimensione Vocabolario: {vocab_size}")

# =================================================================
# 2. CREAZIONE DELLE SEQUENZE (Windowing & Preprocessing)
# =================================================================
# Usiamo la tecnica della "finestra scorrevole" (Sliding Window)
seq_length = 20 # Lunghezza della memoria a breve termine del modello
step = 1        # Spostamento della finestra (1 carattere alla volta)
sequences = []  # Conterrà le finestre di input
next_chars = [] # Conterrà il target (l'etichetta) da predire

for i in range(0, len(text_data) - seq_length, step):
    # Estraiamo i 20 caratteri correnti come input
    sequences.append([char_to_idx[c] for c in text_data[i : i + seq_length]])
    # Il carattere immediatamente successivo è il nostro "target"
    next_chars.append(char_to_idx[text_data[i + seq_length]])

# Trasformazione per Keras: (Numero_Sequenze, Lunghezza_Sequenza, Feature_per_Carattere)
# Dividiamo per vocab_size per normalizzare i dati tra 0 e 1 (aiuta la convergenza del gradiente)
X = np.reshape(sequences, (len(sequences), seq_length, 1)) / float(vocab_size)

# One-Hot Encoding del target: trasforma un numero (es. 5) in un vettore di zeri con un 1 in posizione 5
# Questo è fondamentale per la classificazione multiclasse
y = tf.keras.utils.to_categorical(next_chars, num_classes=vocab_size)

# =================================================================
# 3. DEFINIZIONE DEL MODELLO RNN (Architettura Many-to-One)
# =================================================================


model = Sequential([
    # Layer RNN Ricorrente:
    # 128 unità (neuroni) che mantengono lo stato nascosto.
    # return_sequences=False: significa che la RNN processa l'intera sequenza 
    # e restituisce solo lo stato finale (l'ultimo vettore di contesto).
    layers.SimpleRNN(128, input_shape=(seq_length, 1), return_sequences=False),
    
    # Layer di Output:
    # Un neurone per ogni carattere possibile nel vocabolario.
    # Activation 'softmax': trasforma l'output in una distribuzione di probabilità (somma = 100%)
    layers.Dense(vocab_size, activation='softmax')
])

# Loss: categorical_crossentropy è lo standard quando dobbiamo scegliere una classe tra molte (il carattere giusto)
model.compile(loss='categorical_crossentropy', optimizer='adam')
print(model.summary())

# =================================================================
# 4. FUNZIONE DI GENERAZIONE (Loop di Inferenza)
# =================================================================
def generate_text(seed_text, length=30):
    generated = seed_text
    for _ in range(length):
        # Prendiamo solo gli ultimi 'seq_length' caratteri per la predizione
        input_seq = generated[-seq_length:]
        
        # Trasformiamo i caratteri in numeri e normalizziamo (stesso preprocessing del training)
        x_pred = np.reshape([char_to_idx[c] for c in input_seq], (1, seq_length, 1)) / float(vocab_size)
        
        # Predizione: il modello restituisce le probabilità per ogni carattere del vocabolario
        preds = model.predict(x_pred, verbose=0)[0]
        
        # Argmax: prendiamo l'indice del carattere con la probabilità più alta
        next_index = np.argmax(preds)
        next_char = idx_to_char[next_index]
        
        # Aggiungiamo il carattere predetto alla stringa e continuiamo il loop
        generated += next_char
        
    return generated

# Nota: Senza model.fit(X, y), i pesi sono casuali -> la generazione sarà rumore.
print("\nEsempio di generazione (senza addestramento):")
print(generate_text("l'intelligenza artif"))

Dimensione Vocabolario: 21


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │         2,709 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,349 (75.58 KB)

 Trainable params: 19,349 (75.58 KB)

 Non-trainable params: 0 (0.00 B)

None

Esempio di generazione (senza addestramento):
l'intelligenza artif zz a z  z zeaziz ae z ze a z 


Il testo, per una RNN è una sequenza di scelte probabilistiche, partiamo dalla tokenizzazione per smontare la frase per passare dal dizionario